# 02 — Query Classification

The `QueryClassifier` is the *brain* of the Adaptive RAG pipeline. Every incoming query passes through it, producing a rich classification that tells downstream components:

- **query\_type** — *factual*, *exploratory*, *comparative*, *procedural*, or *opinion*
- **complexity** — *simple*, *moderate*, or *complex*
- **needs\_docs** — should we search the vector store?
- **needs\_web** — should we search the live web?
- **is\_time\_sensitive** — does this query need fresh data?
- **needs\_graph** — does this query need a knowledge graph?

Let's explore each dimension.

## 1. Import the Real Classifier

In [ ]:
import sys
sys.path.insert(0, "..")

try:
    from backend.adaptive_rag.router.query_classifier import QueryClassifier
    from backend.adaptive_rag.router.query_classifier import ClassificationResult
    print("✓ Imported from backend")
except ImportError:
    print("⚠ Backend import failed — please run `pip install -e .` from the project root")
    raise

classifier = QueryClassifier()

## 2. Query Types

The classifier recognises five types based on keyword heuristics:

In [ ]:
queries_by_type = {
    "factual": [
        "What is the boiling point of water?",
        "When was the Eiffel Tower built?",
        "Who invented the telephone?",
    ],
    "exploratory": [
        "Why is the sky blue?",
        "Explain how neural networks work",
        "Describe the process of photosynthesis",
    ],
    "comparative": [
        "Compare React and Vue for frontend development",
        "What is the difference between TCP and UDP?",
        "Python versus Java for data science",
    ],
    "procedural": [
        "How do I deploy a FastAPI application?",
        "Steps to set up a PostgreSQL database",
        "Guide to writing unit tests in Python",
    ],
    "opinion": [
        "What is the best programming language for beginners?",
        "Should I use MongoDB or PostgreSQL?",
        "Recommend a good cloud provider for startups",
    ],
}

print(f"{'Query type':<16} {'Example':<60} {'Detected':<14}")
print("─" * 90)
for qtype, examples in queries_by_type.items():
    for ex in examples:
        r = classifier.classify(ex)
        match = "✓" if r.query_type == qtype else f"→ {r.query_type}"
        print(f"{qtype:<16} {ex[:59]:<60} {match:<14}")
    print()

## 3. Complexity

Complexity is determined by **word count** and **conjunction count** (`and`, `or`, `but`).

| Condition | Result |
|-----------|--------|
| ≤ 10 words, 0 conjunctions | **simple** |
| 11–20 words or 1 conjunction | **moderate** |
| > 20 words or ≥ 2 conjunctions | **complex** |

In [ ]:
print(f"{'Query':<65} {'Words':<6} {'Complexity':<10}")
print("─" * 81)
examples = [
    "What is AI?",
    "Explain how machine learning differs from deep learning",
    "Compare supervised, unsupervised, and reinforcement learning and explain when to use each approach with real-world examples",
]
for q in examples:
    r = classifier.classify(q)
    wc = len(q.split())
    print(f"{q[:64]:<65} {wc:<6} {r.complexity:<10}")

## 4. Boolean Flags

These flags determine which retrieval sources the pipeline should activate:

In [ ]:
test_queries = [
    "What is the population of Japan?",
    "Latest news on climate change",
    "Explain the relationship between genes and personality traits",
]

print(f"{'Query':<60} {'needs_docs':<12} {'needs_web':<12} {'is_time_sensitive':<18} {'needs_graph':<12}")
print("─" * 114)
for q in test_queries:
    r = classifier.classify(q)
    print(f"{q[:59]:<60} {str(r.needs_docs):<12} {str(r.needs_web):<12} {str(r.is_time_sensitive):<18} {str(r.needs_graph):<12}")

## 5. The ClassificationResult Object

`classify()` returns a **pydantic `BaseModel`** — you can access fields as attributes or dump to a dict:

In [ ]:
q = "What are the latest developments in fusion energy research?"
result = classifier.classify(q)

print("Classification result:")
print(f"  query_type       = {result.query_type}")
print(f"  complexity       = {result.complexity}")
print(f"  needs_docs       = {result.needs_docs}")
print(f"  needs_web        = {result.needs_web}")
print(f"  is_time_sensitive= {result.is_time_sensitive}")
print(f"  needs_graph      = {result.needs_graph}")
print()
print("As dictionary:")
import json
print(json.dumps(result.model_dump(), indent=2))

## 6. When Classification Matters Most

| Scenario | Key Signal | Strategy |
|----------|-----------|----------|
| "Latest news on X" | `is_time_sensitive=True` | Web search |
| "Explain Y" + `complexity=complex` | `needs_docs=True` + `needs_web=True` | Hybrid |
| "Define Z" + `complexity=simple` | `needs_docs=True`, `needs_web=False` | Document RAG |
| "What do you think about W" | `query_type=opinion` | Direct LLM |

> **Next:** [03 — Strategy Selection](./03_Strategy_Selection.ipynb) — how the classification maps to strategies